# CROCUS Data on ESS-DIVE

**ESS-DIVE** — the Environmental System Science Data Infrastructure for a Virtual Ecosystem — is a U.S. Department of Energy data repository for Earth and environmental science data. It is used to archive, share, discover, and cite observational, experimental, and modeling datasets from DOE-supported environmental research, including data from ecological, hydrological, geochemical, and urban systems.

The **CROCUS** project (Community Research on Climate and Urban Science) publishes its urban sensor-network data here. This notebook shows how to find those datasets, see what files they contain, and download a dataset's daily files and stitch them into a single NetCDF archive — all without needing to log in.

NetCDF is a common scientific data format for array-oriented data, especially time series, gridded environmental data, and model output. In Python, `xarray` provides a convenient way to open NetCDF files as labeled datasets.

## Browsing the data by hand first

Before writing any code, it helps to see what we're working with. Visit **https://data.ess-dive.lbl.gov/data** and type `CROCUS` in the **Search** box. You'll get a list of datasets. Each one is a *package* with its own **DOI** (a permanent identifier you can cite), and each package holds many individual data files.

The rest of this notebook does the same search programmatically so we can automate the download.

## 1. Setup

Import the libraries we'll use. `requests` talks to the ESS-DIVE web API, and `xarray` reads and combines NetCDF files.

In [32]:
import json
import os
from datetime import datetime, timezone
#from urllib.parse import quote

import pandas as pd
import requests
import xarray as xr

Set up where things live. `BASE` is the ESS-DIVE API address. We keep two folders:

- **`essdive/`** holds the finished, combined archives (the files you'll keep).
- **`essdive/tmp/`** holds the raw daily files we download along the way. These are disposable — once an archive is built you can delete `essdive/tmp/` to reclaim space.

No login or password is needed: the CROCUS datasets we download here are public.

In [33]:
BASE = "https://api.ess-dive.lbl.gov/packages"

OUTDIR  = "essdive"            # final concatenated .nc
WORKDIR = "essdive/tmp"        # disposable daily chunks

os.makedirs(WORKDIR, exist_ok=True)
os.makedirs(OUTDIR, exist_ok=True)

## 2. Search for CROCUS datasets

`search_crocus` asks the ESS-DIVE API for public datasets whose metadata mentions `CROCUS`. The API returns results one page at a time, so the loop below keeps requesting pages until it has collected them all.

Two quirks of this API worth knowing:
- `rowStart` is **1-indexed** — the first result is row 1, not row 0. (Passing 0 returns an error.)
- `isPublic="true"` restricts results to datasets anyone can download.

In [34]:
def search_crocus(text="CROCUS", row_start=1, page_size=25):
    """One page of search results from the ESS-DIVE packages API."""
    params = {"text": text, "isPublic": "true",
              "rowStart": row_start, "pageSize": page_size}
    r = requests.get(BASE, params=params, timeout=60)
    r.raise_for_status()
    return r.json()

# Page through all results
results, start, page_size = [], 1, 25
while True:
    page = search_crocus(row_start=start, page_size=page_size)
    results.extend(page["result"])
    if start + page_size > page["total"]:     # rowStart is 1-indexed
        break
    start += page_size

print(f"Found {page['total']} CROCUS datasets")

Found 28 CROCUS datasets


## 3. Build a manifest of every dataset and its files

The search above only returns *summary* information (title, DOI). To see the actual **files** inside each dataset, we fetch each package individually with `fetch_package`. `build_manifest` does this for every dataset, then caches the result to `essdive_manifest.json` so re-running the notebook doesn't repeatedly hit the API.

For each dataset we record three counts, because a package mixes several kinds of file:
- **`n_entries`** — everything listed, including the metadata XML and PNG quicklook images.
- **`n_files`** — entries that actually have a download link.
- **`n_nc`** — just the NetCDF (`.nc`) data files, which is what we archive.

> **Tip:** if you edit `build_manifest`, delete `essdive_manifest.json` before re-running — otherwise it loads the old cached copy instead of rebuilding.

In [49]:
OBJECT_BASE = "https://data.ess-dive.lbl.gov/catalog/d1/mn/v2/object/"

def fetch_package(doi):
    """Full package JSON (metadata + file list) for one DOI."""
    r = requests.get(f"{BASE}/{doi}", timeout=60)
    r.raise_for_status()
    return r.json()

def file_url(f):
    """Direct download URL for a distribution entry.
    Older ESS-DIVE packages put the URL in contentUrl. Newer ones omit it
    and give only an identifier; the URL is OBJECT_BASE + identifier
    (the same pattern the older packages stored in contentUrl)."""
    if f.get("contentUrl"):
        return f["contentUrl"]
    ident = f.get("identifier")
    return f"{OBJECT_BASE}{ident}" if ident else None

def build_manifest(search_results, cache="essdive_manifest.json"):
    """Fetch every package once, cache to disk, and return a list of
    dicts with the DOI, title, file counts, and downloadable file list."""
    if os.path.exists(cache):
        with open(cache) as f:
            return json.load(f)
    manifest = []
    for i, rec in enumerate(search_results, 1):
        doi = rec["id"]
        pkg = fetch_package(doi)["dataset"]
        files = pkg.get("distribution", []) or []

        dl_files = []
        for f in files:
            url = file_url(f)
            if url:                              # keep entries we can download
                dl_files.append({"name": f.get("name"),
                                 "url": url,
                                 "size": f.get("contentSize"),
                                 "format": f.get("encodingFormat")})
        n_nc = sum(1 for f in dl_files
                   if f["name"] and f["name"].endswith(".nc"))
        manifest.append({
            "doi": doi,
            "title": pkg.get("name"),
            "viewUrl": rec.get("viewUrl"),
            "n_entries": len(files),             # raw distribution count (incl. xml/png)
            "n_files": len(dl_files),            # downloadable files
            "n_nc": n_nc,                        # NetCDF files only
            "files": dl_files,
        })
        print(f"[{i}/{len(search_results)}] {pkg.get('name')[:60]}... "
              f"{n_nc} .nc / {len(dl_files)} files")
    with open(cache, "w") as f:
        json.dump(manifest, f, indent=2)
    return manifest

def find_idx(manifest, *keywords):
    """Return the index of the first dataset whose title contains
    all keywords (case-insensitive). Raises if not found."""
    for i, m in enumerate(manifest):
        title = (m["title"] or "").lower()
        if all(k.lower() in title for k in keywords):
            return i
    raise ValueError(f"No dataset matching {keywords}")

In [50]:
import os
if os.path.exists("essdive_manifest.json"):
    os.remove("essdive_manifest.json")
manifest = build_manifest(results)     # re-fetches with the new file_url logic

[1/28] CROCUS High-Frequency Measurements of CO₂, H₂O, Wind, and Te... 92 .nc / 184 files
[2/28] CROCUS Urban Fluxes of CO₂, H₂O, and Turbulence at Universit... 143 .nc / 285 files
[3/28] CROCUS Weather Data at Northwestern University Rooftop... 252 .nc / 252 files
[4/28] CROCUS Tipping Bucket Rain Gauge Data from Argonne Deployabl... 2 .nc / 2 files
[5/28] CROCUS Tipping Bucket Rain Gauge Data from Argonne Deployabl... 70 .nc / 70 files
[6/28] CROCUS Low Cost All-in-One Weather Station AMB-004 Data Argo... 0 .nc / 0 files
[7/28] CROCUS Tipping Bucket Rain Gauge Data at Argonne National La... 523 .nc / 523 files
[8/28] A network of soil moisture, soil temperature, air temperatur... 0 .nc / 1 files
[9/28] Sap Velocity Data for Urban Trees in Chicago, Illinois (2024... 0 .nc / 1 files
[10/28] Micro Rain Radar Pro Data at the Argonne Testbed for Multisc... 0 .nc / 0 files
[11/28] CROCUS Urban Canyons - Space Science and Engineering Center ... 923 .nc / 923 files
[12/28] CROCUS Low Cost Al

### A note on datasets that show `0 files`

Some datasets print `0 .nc / 0 files`. This does **not** mean they're empty — it means their data is hosted externally (ESS-DIVE's "Tier 2" storage, reached through Globus) rather than served directly by this API. The `download_and_archive` function below only works on datasets whose files have a direct download link. The Tier 2 datasets can still be downloaded by hand from their landing page (the `viewUrl` in the manifest), or via Globus.

So the manifest is your map of *what this notebook can pull directly* versus *what needs the website*.

## 4. Summary: how much data is here?

This totals the downloadable files and their size. **ESS-DIVE reports file sizes in kilobytes (KB)**, so we divide the summed size by 1,000,000 to get gigabytes. (A common mistake is to treat the numbers as bytes, which makes the total look ~1000× too small.)

In [54]:
# Summary before downloading anything
total_nc = sum(m["n_nc"] for m in manifest)
total_files = sum(m["n_files"] for m in manifest)
total_kb = sum(f["size"] or 0 for m in manifest for f in m["files"])
print(f"{len(manifest)} datasets, {total_nc} NetCDF files "
      f"({total_files} downloadable files total), "
      f"{total_kb/1e6:.2f} GB")

28 datasets, 4784 NetCDF files (5064 downloadable files total), 12.97 GB


## 5. Browse the datasets as a table

This turns the manifest into a sortable table so you can decide what to download *before* pulling any data. The **`idx`** column is the number you pass to the download function in the next step. Sizes are converted from KB to MB here.

In [52]:
rows = []
for i, m in enumerate(manifest):
    mb = sum(f["size"] or 0 for f in m["files"]) / 1000   # KB -> MB
    rows.append({"idx": i, "title": m["title"][:55],
                 "nc_files": m["n_nc"], "size_MB": round(mb, 1)})

df = pd.DataFrame(rows).sort_values("size_MB", ascending=False)
pd.set_option("display.max_colwidth", 60)
df[["idx", "title", "nc_files", "size_MB"]]

,idx,title,nc_files,size_MB
0,0,"CROCUS High-Frequency Measurements of CO₂, H₂O, Wind, a",92,9990.3
10,10,CROCUS Urban Canyons - Space Science and Engineering Ce,923,1814.3
7,7,"A network of soil moisture, soil temperature, air tempe",0,318.9
15,15,CROCUS Optical All Precipitation Gauge Data at Argonne,787,182.5
1,1,"CROCUS Urban Fluxes of CO₂, H₂O, and Turbulence at Univ",143,154.6
17,17,CROCUS 3-D wind data from Argonne Deployable Mast durin,48,113.3
8,8,"Sap Velocity Data for Urban Trees in Chicago, Illinois",0,78.8
26,26,CROCUS Weather Data at Northeastern Illinois University,97,63.7
19,19,CROCUS Air Quality Data at Chicago State University Pra,156,62.2
12,12,CROCUS Low Cost All-in-One Weather Station AMB-001 Data,801,46.2


## 6. Download one dataset and build a combined archive

Each CROCUS dataset stores its data as many small NetCDF files — one for every few days. `download_and_archive` downloads all of a dataset's `.nc` files into `essdive/tmp/`, then uses `xarray.open_mfdataset` to stitch them together along the time axis into one file named `<label>_essdive.nc`.

Two features worth pointing out in the code:
- **Resumable downloads.** Each file is written to a temporary `.part` name and only renamed once it's complete, so an interrupted download never leaves a half-written file behind. Re-running skips files already on disk.
- **Provenance.** The combined file records where it came from — the DOI, the landing-page URL, and the date you downloaded it — in its global attributes, so the archive stays self-documenting even if it's copied somewhere else.

Pass the `idx` from the table above and a short `label` (used in the output filename).

In [55]:
def download_and_archive(idx, label, manifest):
    """Download all .nc files for one dataset, concat along time,
    write {label}_essdive.nc with provenance attrs."""
    entry = manifest[idx]
    doi = entry["doi"]
    ncfiles = [f for f in entry["files"] if f["name"].endswith(".nc")]
    print(f"{entry['title'][:55]}: {len(ncfiles)} netCDF files")

    # download each (skip existing, atomic)
    local = []
    sub = os.path.join(WORKDIR, label)
    os.makedirs(sub, exist_ok=True)
    for f in ncfiles:
        dest = os.path.join(sub, f["name"])
        if not (os.path.exists(dest) and os.path.getsize(dest) > 0):
            with requests.get(f["url"], stream=True, timeout=120) as r:
                r.raise_for_status()
                tmp = dest + ".part"
                with open(tmp, "wb") as out:
                    for chunk in r.iter_content(1 << 16):
                        out.write(chunk)
                os.replace(tmp, dest)
        local.append(dest)

    # concat along time
    ds = xr.open_mfdataset(sorted(local), combine="by_coords")

    # provenance
    ds.attrs["source_archive"] = f"ESS-DIVE {doi}"
    ds.attrs["source_url"] = entry["viewUrl"]
    ds.attrs["essdive_retrieved"] = datetime.now(timezone.utc).isoformat()
    ds.attrs["essdive_n_files"] = len(ncfiles)

    out = os.path.join(OUTDIR, f"{label}_essdive.nc")
    ds.to_netcdf(out)
    ds.close()
    print(f"wrote {out}  ({os.path.getsize(out)/1e6:.1f} MB)")
    return out

### Example: the NEIU rooftop weather and air-quality datasets

Run the cell below to build two archives — the Northeastern Illinois University rooftop weather station (WXT) and air-quality sensor (AQT). Confirm the `idx` values still point at the NEIU datasets in the table above first; they can shift if ESS-DIVE adds new packages.

In [57]:
wxt_idx = find_idx(manifest, "Northwestern", "Weather")
#aqt_idx = find_idx(manifest, "Northwestern", "Air Quality")

print(wxt_idx, manifest[wxt_idx]["title"])
#print(aqt_idx, manifest[aqt_idx]["title"])


2 CROCUS Weather Data at Northwestern University Rooftop


In [ ]:

download_and_archive(wxt_idx, "NU_wxt", manifest)
#download_and_archive(aqt_idx, "NU_aqt", manifest)

CROCUS Weather Data at Northwestern University Rooftop: 252 netCDF files


## 7. Open and inspect a finished archive

Once an archive is written, open it back up to check what's inside — the variables, the time range, and the provenance attributes we attached. Change the filename to inspect a different archive.

In [14]:
ds = xr.open_dataset(os.path.join(ARCHIVE, "NEIU_wxt_essdive.nc"))
ds

<xarray.Dataset> Size: 64MB
Dimensions:        (time: 797641)
Coordinates:
  * time           (time) datetime64[ns] 6MB 2023-05-16T21:30:10 ... 2023-08-...
Data variables:
    temperature    (time) float64 6MB ...
    humidity       (time) float64 6MB ...
    pressure       (time) float64 6MB ...
    rainfall       (time) float64 6MB ...
    dewpoint       (time) float64 6MB ...
    wetbulb        (time) float64 6MB ...
    wind_dir_10s   (time) float64 6MB ...
    wind_mean_10s  (time) float64 6MB ...
    wind_max_10s   (time) float64 6MB ...
Attributes:
    conventions:        CF 1.10
    site_ID:            NEIU
    node:               W08D
    CAMS_tag:           CMS-WXT-002
    datastream:         CMS_wxt536_NEIU_a1
    datalevel:          a1
    latitude:           41.9804526
    longitude:          -87.7196038
    source_archive:     ESS-DIVE ess-dive-b43a0b35cacbef1-20240501T175506250
    source_url:         https://data.ess-dive.lbl.gov/view/doi:10.15485/2338027
    essdive_retrieved:  2026-06-25T14:14:39.364319+00:00
    essdive_n_files:    97